In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
fs = pd.read_csv("output/feasibility_study.csv")
fs

In [ ]:


conf = pd.read_excel("confounders.xlsx", header=0, index_col=0)
conf.columns = ["miner", "file", "n_base", "n_res", "n_alt_util", "n_confounders"]
conf["file"] = conf["file"].str.removesuffix(".exs")

df = fs.merge(conf[["file", "n_confounders"]], on="file", how="left")

count_cols = ["n_cases", "n_executions", "n_decision_rows", "n_choice_sets", "n_transitions_fitted", "n_confounders"]
time_cols  = ["t_util_s", "t_dml_s", "t_forest_s", "t_backdoor_s"]

for c in count_cols + time_cols:
    df[c] = pd.to_numeric(df[c], errors="coerce")

print(f"Rows total: {len(df)}  |  rows with n_confounders: {df['n_confounders'].notna().sum()}")
df[count_cols + time_cols].describe()

In [ ]:
# Pairwise Pearson correlation: rows = times, columns = counts
corr = pd.DataFrame(
    {count: {time: df[[count, time]].dropna().corr().loc[count, time]
             for time in time_cols}
     for count in count_cols},
    index=time_cols
)[count_cols]

corr

In [ ]:
corr.to_csv('correlations.csv')

In [ ]:
fig, ax = plt.subplots(figsize=(10, 3.5))
sns.heatmap(
    corr,
    annot=True, fmt=".2f",
    vmin=-1, vmax=1, center=0,
    cmap="RdBu_r",
    linewidths=0.5,
    ax=ax,
)
ax.set_title("Pairwise Pearson correlation — size metrics vs. computation times", pad=12)
ax.set_xlabel("")
ax.set_ylabel("")
plt.tight_layout()
plt.savefig("output/correlation_matrix.pdf", bbox_inches="tight")
plt.show()
print("Saved → output/correlation_matrix.pdf")